# 10 — SHAP Interpretability for LSTM Feature Attribution

**Goal:** Use SHAP to understand what drives the Optuna-tuned LSTM's predictions, and compare feature importance across high-volatility vs low-volatility market regimes.

**Explainer:** Tries `shap.DeepExplainer` first, falls back to `GradientExplainer`, then `KernelExplainer` (model-agnostic). On TF 2.21, KernelExplainer is used due to a gradient-registry incompatibility.

In [ ]:
import sys, pickle, json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'Data' / 'cryptonews.csv').exists() or (ROOT / 'notebooks').exists():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

INTERIM = ROOT / 'notebooks' / 'interim'
OUTPUTS = ROOT / 'outputs'

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# Load features and model
with (INTERIM / 'features_for_lstm.pkl').open('rb') as f:
    bundle = pickle.load(f)

train_x = bundle['train_x']
test_x = bundle['test_x']
test_close = bundle['test_close']
feature_names = bundle['feature_cols']

model = tf.keras.models.load_model(INTERIM / 'best_optuna_model.keras')
print(f'Model: {model.name}')
print(f'Train: {train_x.shape}  Test: {test_x.shape}  Features: {len(feature_names)}')

## 10.1 Compute SHAP values
DeepExplainer → GradientExplainer → KernelExplainer fallback chain.

In [ ]:
from src.interpretability.shap_explainer import run_shap_analysis

result = run_shap_analysis(
    model=model,
    train_x=train_x,
    test_x=test_x,
    feature_names=feature_names,
    test_close=test_close,
    output_dir=OUTPUTS,
)
print(f'\nExplainer used: {result["explainer_used"]}')
print(f'SHAP values shape: {result["shap_values_shape"]}')
print(f'Regime split: {result["regime_info"]}')

## 10.2 Global feature importance ranking

In [ ]:
importance = pd.read_csv(OUTPUTS / 'shap_feature_importance.csv')
importance.head(15)

## 10.3 SHAP summary plot (beeswarm)
Shows the global feature importance and direction of impact.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUTPUTS / 'shap_summary.png')))

## 10.4 Regime comparison: High-Vol vs Low-Vol
Split test days by the median 20-day realized vol. Does the model rely on different features in different regimes?

In [ ]:
display(Image(filename=str(OUTPUTS / 'shap_regime_comparison.png')))

## 10.5 Summary
- SHAP values computed via KernelExplainer (DeepExplainer and GradientExplainer fail on TF 2.21 due to a gradient-registry incompatibility).
- Top features by mean |SHAP|: `llm_pos_share_lag5`, `pos_share`, `macd_hist`, `news_count`, `neg_share`.
- The model relies on both technical indicators (MACD, Bollinger width, RSI) and sentiment features (LLM sentiment lags, positive/negative share).
- Regime comparison reveals whether the model shifts its reliance between technical and sentiment features across volatility regimes.
- Outputs: `outputs/shap_summary.png`, `outputs/shap_regime_comparison.png`, `outputs/shap_feature_importance.csv`.